# Wage Model — End-to-End Training, Evaluation, SHAP and Robustness

This is the second notebook in the project. It consumes the harmonized dataset
produced by `01_wage_merge.ipynb`, trains the quantile MLP, evaluates it on a
held-out test split, runs SHAP / permutation / ablation and robustness checks,
and writes every figure used in the slide deck.

**Input:** `data/processed/plfs_2023_2024_2025_harmonized.csv` (213,241 rows)

**Outputs:**

- `models/wage/wage_quantile_mlp.pt` — trained weights
- `models/wage/wage_preprocessing.pkl` — preprocessing bundle
- `models/wage/model_info.json` — model metadata
- `models/wage/metrics.json` — evaluation, SHAP and robustness results
- `models/wage/test_predictions.csv.gz` — held-out predictions
- `models/wage/shap_importance.csv`, `shap_permutation.csv`, `shap_ablation.csv`
- `models/wage/wage_perturbations.csv`, `wage_subgroups.csv`, `wage_drift_summary.json`
- `figures/training/*.png` — training and test-evaluation figures
- `figures/presentation/*.png` — metrics, SHAP, robustness and drift figures

The model is retrained from scratch by this notebook. The metrics should be
close to the documented final model but may differ slightly across hardware.
A CUDA GPU is recommended. The SHAP section is the slowest part; reduce
`SHAP_N_TEST` / `SHAP_NSAMPLES` at the top for a quick run.


## 1. Configuration

Paths are resolved relative to the project root folder, so the notebook works
from either the project root or `notebooks/`. All randomness is seeded
for reproducibility.

In [ ]:
import os
import json
import math
import pickle
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path.cwd()
while not (ROOT / "notebooks").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "notebooks").exists(), f"repository root not found from {Path.cwd()}"

DATA_PATH = ROOT / "data" / "processed" / "plfs_2023_2024_2025_harmonized.csv"
OUT_DIR = ROOT / "models" / "wage"
FIG_TRAIN = ROOT / "figures" / "training"
FIG_PRES = ROOT / "figures" / "presentation"
for _d in (OUT_DIR, FIG_TRAIN, FIG_PRES):
    _d.mkdir(parents=True, exist_ok=True)

TARGET = "ern_reg_current"

WEIGHTS_PATH = OUT_DIR / "wage_quantile_mlp.pt"
BUNDLE_PATH = OUT_DIR / "wage_preprocessing.pkl"
MODEL_INFO_PATH = OUT_DIR / "model_info.json"
METRICS_PATH = OUT_DIR / "metrics.json"
PREDICTIONS_PATH = OUT_DIR / "test_predictions.csv.gz"

NUMERIC_FEATURES = [
    "age", "potential_experience", "potential_experience_sq",
    "years_formal_education", "tothrs_wrk", "days_worked", "avg_daily_hours",
]
CATEGORICAL_FEATURES = [
    "sex", "marst", "rel", "gedu_lvl", "tedu_lvl", "curr_att", "secondary",
    "voc", "voc_fld", "voc_typ", "st", "dc", "state_district", "sec",
    "ind_pas", "ocu_pas", "loc_pas", "etyp_pas", "wrkr_pas", "job_pas",
    "leave_pas", "ssec_pas", "has_sas", "acws", "aind_cws", "ocu_cws", "dur_pas",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
QUANTILES = [0.05, 0.50, 0.95]

BATCH_SIZE = 4096
MAX_EPOCHS = int(os.environ.get("WAGE_EPOCHS", "50"))
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

SHAP_N_TEST = int(os.environ.get("SHAP_N_TEST", "2000"))
SHAP_N_BG = int(os.environ.get("SHAP_N_BG", "250"))
SHAP_NSAMPLES = int(os.environ.get("SHAP_NSAMPLES", "150"))
N_BOOT = int(os.environ.get("N_BOOT", "500"))
MC_PASSES = int(os.environ.get("MC_PASSES", "10"))

print("Dataset:", DATA_PATH)
print("Model outputs:", OUT_DIR)
print("Figures:", FIG_TRAIN, "|", FIG_PRES)
print("Epochs:", MAX_EPOCHS, "(override with WAGE_EPOCHS for smoke tests)")
print("SHAP sample:", SHAP_N_TEST, "test rows /", SHAP_N_BG, "background /", SHAP_NSAMPLES, "nsamples")

## 2. Load and validate the model-ready dataset

The merge notebook is the single source of truth. These guards fail loudly if
the dataset drifts: row counts, the 69 retained duplicate rows, positive
target, and the year-specific education quirks that the deployed model was
trained with.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

EXPECTED_ROWS = {2023: 45569, 2024: 46034, 2025: 121638}
assert len(df) == 213241, f"unexpected row count: {len(df)}"
assert df.shape[1] == 41, f"unexpected column count: {df.shape[1]}"
assert int(df.duplicated().sum()) == 69, int(df.duplicated().sum())
assert df[TARGET].notna().all() and (df[TARGET] > 0).all()
assert set(FEATURES).issubset(df.columns)

for year, expected in EXPECTED_ROWS.items():
    n = int((df["survey_year"] == year).sum())
    assert n == expected, f"{year}: {n} rows, expected {expected}"

# 2025 education quirk from the original benchmark: only codes 10-13 map.
yrs25 = df.loc[df["survey_year"] == 2025, "years_formal_education"]
assert int(yrs25.isna().sum()) == 51735, int(yrs25.isna().sum())
assert set(yrs25.dropna().unique()) == {15.0, 17.0}, sorted(yrs25.dropna().unique())

df = df[df[TARGET].notna() & (df[TARGET] > 0)].copy().reset_index(drop=True)

summary = df.groupby("survey_year").agg(
    rows=(TARGET, "size"),
    wage_median=(TARGET, "median"),
    wage_mean=(TARGET, "mean"),
)
print("Validated dataset:", df.shape)
print(summary.to_string())
print("\nRetained exact duplicate rows:", int(df.duplicated().sum()))
print("Education NaNs by year:")
print(df.groupby("survey_year")["years_formal_education"].apply(lambda s: int(s.isna().sum())).to_string())

## 3. Stratified 70 / 15 / 15 split

The split is stratified by `survey_year x wage band` (<=10k, 10-20k, 20-50k,
50-100k, >100k) with seed 42, matching the deployed training protocol. The
same row order as the source CSV is preserved, so the split is reproducible.

In [ ]:
def wage_group(x):
    if x <= 10000:
        return "0_10k"
    if x <= 20000:
        return "10_20k"
    if x <= 50000:
        return "20_50k"
    if x <= 100000:
        return "50_100k"
    return "100k_plus"


df["_stratum"] = df["survey_year"].astype(str) + "_" + df[TARGET].apply(wage_group)

all_idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.30, random_state=SEED, stratify=df["_stratum"]
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED,
    stratify=df.iloc[temp_idx]["_stratum"],
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print("\nTest rows by year:")
print(test_df.groupby("survey_year").size().to_string())
print("\nTest rows by stratum:")
print(test_df["_stratum"].value_counts().sort_index().to_string())

## 4. Preprocessing

Numeric features: median imputation + standardisation, fit on train only.
Categorical features: most-frequent imputation + ordinal encoding with
`unknown_value=-1`, then shifted by +1 so index 0 is reserved for padding.
Embedding widths are `min(50, max(2, cardinality // 2))`.

In [ ]:
def fit_transform_benchmark(train_part, val_part, test_part):
    num_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(num_imputer.fit_transform(train_part[NUMERIC_FEATURES]))
    X_val_num = scaler.transform(num_imputer.transform(val_part[NUMERIC_FEATURES]))
    X_test_num = scaler.transform(num_imputer.transform(test_part[NUMERIC_FEATURES]))

    cat_imputer = SimpleImputer(strategy="most_frequent")
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    X_train_cat = encoder.fit_transform(
        cat_imputer.fit_transform(train_part[CATEGORICAL_FEATURES].astype(str))
    ).astype(np.int64) + 1
    X_val_cat = encoder.transform(
        cat_imputer.transform(val_part[CATEGORICAL_FEATURES].astype(str))
    ).astype(np.int64) + 1
    X_test_cat = encoder.transform(
        cat_imputer.transform(test_part[CATEGORICAL_FEATURES].astype(str))
    ).astype(np.int64) + 1

    embedding_sizes = [
        (len(categories) + 1, min(50, max(2, (len(categories) + 1) // 2)))
        for categories in encoder.categories_
    ]

    return {
        "train_num": X_train_num,
        "val_num": X_val_num,
        "test_num": X_test_num,
        "train_cat": X_train_cat,
        "val_cat": X_val_cat,
        "test_cat": X_test_cat,
        "train_y": np.log1p(train_part[TARGET].to_numpy(dtype=np.float32)),
        "val_y": np.log1p(val_part[TARGET].to_numpy(dtype=np.float32)),
        "test_y": test_part[TARGET].to_numpy(dtype=np.float32),
        "embedding_sizes": embedding_sizes,
        "numeric_imputer": num_imputer,
        "numeric_scaler": scaler,
        "categorical_imputer": cat_imputer,
        "categorical_encoder": encoder,
    }


prep = fit_transform_benchmark(train_df, val_df, test_df)


class WageDataset(Dataset):
    def __init__(self, categorical, numeric, target):
        self.categorical = categorical
        self.numeric = numeric
        self.target = target

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return self.categorical[idx], self.numeric[idx], self.target[idx]


def make_loader(cat, num, tgt, shuffle):
    return DataLoader(
        WageDataset(
            torch.tensor(cat, dtype=torch.long),
            torch.tensor(num, dtype=torch.float32),
            torch.tensor(tgt, dtype=torch.float32),
        ),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


train_loader = make_loader(prep["train_cat"], prep["train_num"], prep["train_y"], True)
val_loader = make_loader(prep["val_cat"], prep["val_num"], prep["val_y"], False)
test_loader = make_loader(prep["test_cat"], prep["test_num"], prep["test_y"], False)

print("Embeddings:", sum(n for n, _ in prep["embedding_sizes"]), "total rows")
print("Largest categorical cardinalities:")
for name, (n, d) in sorted(
    zip(CATEGORICAL_FEATURES, prep["embedding_sizes"]), key=lambda kv: -kv[1][0]
)[:5]:
    print(f"  {name:16s} {n:6d} categories -> dim {d}")
print("Preprocessing complete. DataLoaders ready.")

## 5. Model and weighted pinball loss

`TabularEncoder` embeds each categorical column, concatenates the numeric
block, projects to 256 dimensions, then applies four residual blocks.
`QuantileRegressor` emits three quantiles in log1p space.

The loss is the sum of the three pinball losses, with a 5x multiplier for
targets above Rs 100,000 (in log1p space) so the upper tail is not
under-weighted.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, size, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(size, size), nn.LayerNorm(size), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(size, size), nn.LayerNorm(size),
        )

    def forward(self, x):
        return nn.GELU()(x + self.block(x))


class TabularEncoder(nn.Module):
    def __init__(self, embedding_sizes, numeric_size):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(n, d) for n, d in embedding_sizes])
        total_emb_size = sum(d for _, d in embedding_sizes)
        self.input_layer = nn.Sequential(
            nn.Linear(total_emb_size + numeric_size, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.15),
        )
        self.residual_blocks = nn.Sequential(
            ResidualBlock(256), ResidualBlock(256), ResidualBlock(256), ResidualBlock(256)
        )

    def forward(self, categorical, numeric):
        embs = [emb(categorical[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embs + [numeric], dim=1)
        return self.residual_blocks(self.input_layer(x))


class QuantileRegressor(nn.Module):
    def __init__(self, embedding_sizes, numeric_size):
        super().__init__()
        self.encoder = TabularEncoder(embedding_sizes, numeric_size)
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 32), nn.GELU(), nn.Linear(32, 3),
        )

    def forward(self, categorical, numeric):
        return self.head(self.encoder(categorical, numeric))


HIGH_INCOME_LOG = math.log1p(100000.0)


def weighted_pinball_loss(predictions, target):
    target_exp = target.unsqueeze(1)
    losses = []
    for i, q in enumerate(QUANTILES):
        error = target_exp - predictions[:, i:i + 1]
        losses.append(torch.max((q - 1) * error, q * error))
    raw_loss = torch.cat(losses, dim=1).sum(dim=1)
    weights = torch.where(target > HIGH_INCOME_LOG, 5.0, 1.0)
    return (raw_loss * weights).mean()


def pinball_np(y_true, y_pred, q):
    error = y_true - y_pred
    return float(np.mean(np.maximum((q - 1) * error, q * error)))

## 6. Train

AdamW, 50 epochs, batch 4096, lr 1e-3, weight decay 1e-5. The epoch with the
lowest validation weighted pinball loss is restored at the end.

In [ ]:
print("Training quantile MLP...")
model = QuantileRegressor(prep["embedding_sizes"], len(NUMERIC_FEATURES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_loss = float("inf")
best_epoch = None
best_state = None
history = []
t_start = time.time()

for epoch in range(MAX_EPOCHS):
    model.train()
    train_total = 0.0
    for cat, num, tgt in train_loader:
        cat = cat.to(device, non_blocking=True)
        num = num.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = weighted_pinball_loss(model(cat, num), tgt)
        loss.backward()
        optimizer.step()
        train_total += loss.item() * len(tgt)

    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for cat, num, tgt in val_loader:
            cat = cat.to(device, non_blocking=True)
            num = num.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            val_total += weighted_pinball_loss(model(cat, num), tgt).item() * len(tgt)

    train_loss = train_total / len(train_loader.dataset)
    val_loss = val_total / len(val_loader.dataset)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = " *"

    print(f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} train {train_loss:.5f} val {val_loss:.5f}{marker}")

model.load_state_dict(best_state)
model.eval()
print("=" * 72)
print(f"Best epoch {best_epoch} - validation loss {best_val_loss:.5f}")
print(f"Training time: {(time.time() - t_start) / 60:.1f} min")
print("=" * 72)

## 7. Evaluation and calibration analysis

Held-out metrics: 5-95 coverage, median MAE and bias, interval width, quantile
crossings, per-quantile pinball loss and empirical coverage, plus slices by
survey year and wage band. The 2025 slice matters because the original
harmonisation left most 2025 education values missing; the model must remain
stable on it.

In [ ]:
def predict_quantiles(model, loader):
    parts = []
    with torch.no_grad():
        for cat, num, _ in loader:
            parts.append(model(cat.to(device), num.to(device)).cpu().numpy())
    predictions = np.concatenate(parts, axis=0)
    predictions = np.maximum(np.expm1(predictions), 0)
    return predictions[:, 0], predictions[:, 1], predictions[:, 2]


test_lower, test_median, test_upper = predict_quantiles(model, test_loader)
test_actual = prep["test_y"]

inside = (test_actual >= test_lower) & (test_actual <= test_upper)
width = test_upper - test_lower
median_abs_error = np.abs(test_actual - test_median)
median_bias = test_median - test_actual
crossings = (test_lower > test_median) | (test_median > test_upper) | (test_lower > test_upper)
parameter_count = sum(p.numel() for p in model.parameters())


def slice_metrics(mask, label):
    n = int(mask.sum())
    if n == 0:
        return {"label": label, "count": 0}
    return {
        "label": label,
        "count": n,
        "coverage_5_95": float(inside[mask].mean()),
        "mae": float(np.mean(median_abs_error[mask])),
        "bias": float(np.mean(median_bias[mask])),
        "median_width": float(np.median(width[mask])),
        "mean_width": float(np.mean(width[mask])),
        "crossing_rate": float(crossings[mask].mean()),
        "pinball_05": pinball_np(test_actual[mask], test_lower[mask], 0.05),
        "pinball_50": pinball_np(test_actual[mask], test_median[mask], 0.50),
        "pinball_95": pinball_np(test_actual[mask], test_upper[mask], 0.95),
        "empirical_coverage_05": float((test_actual[mask] <= test_lower[mask]).mean()),
        "empirical_coverage_50": float((test_actual[mask] <= test_median[mask]).mean()),
        "empirical_coverage_95": float((test_actual[mask] <= test_upper[mask]).mean()),
    }


overall = slice_metrics(np.ones(len(test_actual), dtype=bool), "all")
print("FINAL TEST EVALUATION")
print(f"  Rows:                  {overall['count']:,}")
print(f"  5-95 coverage:         {overall['coverage_5_95']:.2%}")
print(f"  Median MAE:            Rs {overall['mae']:,.2f}")
print(f"  Median bias:           Rs {overall['bias']:,.2f}")
print(f"  Median interval width: Rs {overall['median_width']:,.2f}")
print(f"  Mean interval width:   Rs {overall['mean_width']:,.2f}")
print(f"  Quantile crossings:    {int(crossings.sum())} ({overall['crossing_rate']:.4%})")
print(f"  Pinball 05/50/95:      {overall['pinball_05']:.2f} / {overall['pinball_50']:.2f} / {overall['pinball_95']:.2f}")
print(f"  Empirical coverage:    {overall['empirical_coverage_05']:.2%} / "
      f"{overall['empirical_coverage_50']:.2%} / {overall['empirical_coverage_95']:.2%}")

test_years = test_df["survey_year"].to_numpy()
year_metrics = {}
print("\nBY SURVEY YEAR")
for year in sorted(np.unique(test_years)):
    m = slice_metrics(test_years == year, str(int(year)))
    year_metrics[str(int(year))] = m
    print(f"  {int(year)}: n={m['count']:,} coverage={m['coverage_5_95']:.2%} "
          f"MAE=Rs {m['mae']:,.0f} bias=Rs {m['bias']:,.0f} "
          f"pinball50={m['pinball_50']:.1f}")

band_edges = [0, 10000, 20000, 50000, 100000, np.inf]
band_labels = ["0_10k", "10_20k", "20_50k", "50_100k", "100k_plus"]
band_metrics = {}
print("\nBY WAGE BAND")
for i, label in enumerate(band_labels):
    m = slice_metrics(
        (test_actual > band_edges[i]) & (test_actual <= band_edges[i + 1]), label
    )
    band_metrics[label] = m
    print(f"  {label:9s}: n={m['count']:,} coverage={m['coverage_5_95']:.2%} "
          f"MAE=Rs {m['mae']:,.0f} bias=Rs {m['bias']:,.0f}")

high_income = {}
print("\nHIGH-INCOME SLICES")
for label, threshold in [(">50000", 50000), (">100000", 100000)]:
    m = slice_metrics(test_actual > threshold, label)
    high_income[label] = m
    print(f"  {label:8s}: n={m['count']:,} coverage={m['coverage_5_95']:.2%} "
          f"MAE=Rs {m['mae']:,.0f} median_width=Rs {m['median_width']:,.0f}")

## 8. Export artifacts

Weights, preprocessing bundle and metadata are written to `models/wage/`.
The bundle contains every object required for inference, so a consumer never
needs to refit preprocessing. `metrics.json` records the split sizes, training
history, calibration analysis and all slices.

In [ ]:
model.eval()
torch.save(model.state_dict(), WEIGHTS_PATH)

bundle = {
    "model_type": "PLFS 2023-2025 weighted 5-50-95 quantile MLP",
    "target": TARGET,
    "quantiles": QUANTILES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "features": FEATURES,
    "feature_count": len(FEATURES),
    "removed_features": ["qtr", "nss_reg", "sro"],
    "embedding_sizes": prep["embedding_sizes"],
    "numeric_imputer": prep["numeric_imputer"],
    "numeric_scaler": prep["numeric_scaler"],
    "categorical_imputer": prep["categorical_imputer"],
    "categorical_encoder": prep["categorical_encoder"],
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_val_loss),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "calibration": None,
    "production_type": "WEIGHTED_BENCHMARK_MODEL",
    "preprocessing_fit_on": "TRAIN_ONLY",
}

with open(BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

model_info = {
    "model_type": bundle["model_type"],
    "years": [2023, 2024, 2025],
    "feature_count": len(FEATURES),
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "removed_features": bundle["removed_features"],
    "quantiles": QUANTILES,
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_val_loss),
    "parameter_count": int(parameter_count),
    "calibration": None,
    "production_type": bundle["production_type"],
    "preprocessing_fit_on": bundle["preprocessing_fit_on"],
    "dataset": {
        "path": str(DATA_PATH.relative_to(ROOT)).replace("\\", "/"),
        "rows": int(len(df)),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "duplicate_rows_retained": int(df.duplicated().sum()),
    },
    "test_coverage": float(overall["coverage_5_95"]),
    "test_median_prediction_mae": float(overall["mae"]),
    "test_median_absolute_error": float(np.median(median_abs_error)),
    "test_median_prediction_bias": float(overall["bias"]),
    "test_median_interval_width": float(overall["median_width"]),
    "test_mean_interval_width": float(overall["mean_width"]),
    "test_quantile_crossings": int(crossings.sum()),
}

with open(MODEL_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

metrics = {
    "dataset": model_info["dataset"],
    "config": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "quantiles": QUANTILES,
        "high_income_weight": 5.0,
        "high_income_threshold": 100000.0,
    },
    "training": {
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_val_loss),
        "history": history,
    },
    "test": {
        "overall": overall,
        "by_year": year_metrics,
        "by_wage_band": band_metrics,
        "high_income": high_income,
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

predictions = pd.DataFrame({
    "survey_year": test_df["survey_year"].to_numpy(),
    "actual": test_actual,
    "q05": test_lower,
    "q50": test_median,
    "q95": test_upper,
    "inside_5_95": inside,
    "interval_width": width,
})
predictions.to_csv(PREDICTIONS_PATH, index=False, compression="gzip")

print("Export complete:")
for p in (WEIGHTS_PATH, BUNDLE_PATH, MODEL_INFO_PATH, METRICS_PATH, PREDICTIONS_PATH):
    print(f"  {p.name:70s} {p.stat().st_size / 1e6:8.2f} MB")

## 9. Summary of the training run

- Validated the model-ready dataset against the 01 merge contract.
- Trained the quantile MLP with a stratified 70/15/15 split.
- Evaluated 5-95 coverage, pinball loss, crossing rate and year/wage-band slices.
- Exported the weights, preprocessing bundle, model info, metrics and test predictions.

The next sections add SHAP, robustness, schema-drift and presentation figures.


## 10. Training and test-evaluation figures

These figures use the training history and held-out predictions from this run.
They are written to `figures/training/`.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_ROOTS = Path.cwd()
while not (_ROOTS / "notebooks").exists() and _ROOTS.parent != _ROOTS:
    _ROOTS = _ROOTS.parent
WAGE_OUT = _ROOTS / "models" / "wage"
PLOT_DIR = _ROOTS / "figures" / "training"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
C_BLUE, C_ORANGE, C_GREEN, C_RED = "#4C78A8", "#F58518", "#54A24B", "#E45756"

metrics = json.loads((WAGE_OUT / "metrics.json").read_text(encoding="utf-8"))
pred = pd.read_csv(WAGE_OUT / "test_predictions.csv.gz")
hist = pd.DataFrame(metrics["training"]["history"])
best_epoch = int(metrics["training"]["best_epoch"])

# 1) Training / validation loss.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hist["epoch"], hist["train_loss"], lw=2, color=C_BLUE, label="train")
ax.plot(hist["epoch"], hist["val_loss"], lw=2, color=C_ORANGE, label="validation")
ax.axvline(best_epoch, color="black", ls="--", lw=1)
ax.annotate(f"best epoch {best_epoch}", xy=(best_epoch, hist["val_loss"].min()),
            xytext=(8, 18), textcoords="offset points", fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_title("Quantile MLP — training history")
ax.set_xlabel("epoch")
ax.set_ylabel("weighted pinball loss (log scale)")
ax.set_yscale("log")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "01_training_history.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved:", PLOT_DIR / "01_training_history.png")


In [ ]:
# 2) Headline test metrics table.
def block(mask, label):
    d = pred[mask]
    return [
        label, f"{len(d):,}",
        f"{100 * d['inside_5_95'].mean():.2f}%",
        f"Rs {d['actual'].sub(d['q50']).abs().mean():,.0f}",
        f"Rs {d['q50'].sub(d['actual']).mean():,.0f}",
        f"Rs {d['interval_width'].median():,.0f}",
    ]

rows = [
    block(pd.Series(True, index=pred.index), "All"),
    block(pred["actual"] > 50_000, "> Rs 50k"),
    block(pred["actual"] > 100_000, "> Rs 100k"),
]
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis("off")
tbl = ax.table(cellText=rows, colLabels=["Slice", "n", "5-95 coverage", "MAE", "Bias", "Median width"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D9D9D9")
    if r == 0:
        cell.set_facecolor("#EAF2F8"); cell.set_text_props(weight="bold")
ax.set_title("Final wage model — held-out test metrics", pad=12)
fig.tight_layout()
fig.savefig(PLOT_DIR / "02_test_metrics_table.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# 3) Calibration by predicted-median decile.
d = pred.copy()
d["decile"] = pd.qcut(d["q50"], q=10, labels=False, duplicates="drop")
rel = d.groupby("decile").agg(
    mean_pred=("q50", "mean"),
    mean_actual=("actual", "mean"),
    coverage=("inside_5_95", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rel["mean_pred"], rel["mean_actual"], marker="o", lw=2, color=C_BLUE)
lo = min(rel["mean_pred"].min(), rel["mean_actual"].min())
hi = max(rel["mean_pred"].max(), rel["mean_actual"].max())
axes[0].plot([lo, hi], [lo, hi], "--", color="black", lw=1)
axes[0].set_xlabel("mean predicted median wage (Rs)")
axes[0].set_ylabel("mean actual wage (Rs)")
axes[0].set_title("Calibration by predicted decile")
axes[0].grid(alpha=0.25)
axes[0].spines[["top", "right"]].set_visible(False)
axes[1].bar([str(int(x) + 1) for x in rel["decile"]], 100 * rel["coverage"], color=C_GREEN)
axes[1].axhline(90, color="black", ls="--", lw=1)
axes[1].set_xlabel("predicted-median decile")
axes[1].set_ylabel("5-95 coverage (%)")
axes[1].set_title("Coverage by predicted decile")
axes[1].set_ylim(75, 100)
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "03_calibration_by_decile.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# 4) Coverage by actual wage band.
bands = pd.cut(pred["actual"], bins=[-1, 10_000, 20_000, 50_000, 100_000, np.inf],
               labels=["0-10k", "10-20k", "20-50k", "50-100k", "100k+"])
cov = pred.groupby(bands, observed=True)["inside_5_95"].mean()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cov.index.astype(str), 100 * cov.values, color=C_ORANGE, width=0.55)
for b, v in zip(bars, 100 * cov.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.4, f"{v:.1f}%", ha="center", fontsize=10)
ax.axhline(90, color="black", ls="--", lw=1, label="nominal 90%")
ax.set_ylim(75, 100)
ax.set_ylabel("5-95 coverage (%)")
ax.set_title("Coverage by actual wage band")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "04_coverage_by_wage_band.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved:", PLOT_DIR / "02_test_metrics_table.png")
print("Saved:", PLOT_DIR / "03_calibration_by_decile.png")
print("Saved:", PLOT_DIR / "04_coverage_by_wage_band.png")


## 11. SHAP, permutation and ablation

The model has 7 numeric inputs and 27 categorical inputs. For SHAP we flatten
the numeric block and the concatenated embedding vectors, run GradientSHAP,
then sum the embedding dimensions back into one attribution per original
categorical feature.

GradientSHAP is an approximation to exact Shapley values. Two independent
checks are computed:

- **permutation importance**: shuffle one raw column and measure the MAE change;
- **ablation importance**: replace one column with its training median/mode and
  measure the MAE change.

The sample sizes are set in the configuration cell. Reduce them for a quicker
run; increase them for a final analysis.


In [ ]:
# ---------------------------------------------------------------------------
# Inference helpers used by SHAP, robustness and drift.
# ---------------------------------------------------------------------------
import shap

HEADS = {0: "q05", 1: "q50", 2: "q95"}
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def to_py(obj):
    if isinstance(obj, dict):
        return {str(k): to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_py(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, float):
        return None if not np.isfinite(obj) else obj
    return obj


def dump_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_py(obj), indent=2, ensure_ascii=False), encoding="utf-8")


def encode_frame(frame):
    cat = frame[CATEGORICAL_FEATURES].astype(str)
    Xc = prep["categorical_encoder"].transform(
        prep["categorical_imputer"].transform(cat)
    ).astype(np.int64) + 1
    Xn = prep["numeric_scaler"].transform(
        prep["numeric_imputer"].transform(frame[NUMERIC_FEATURES])
    ).astype(np.float32)
    return Xc, Xn


def predict_log_frame(frame):
    Xc, Xn = encode_frame(frame)
    chunks = []
    with torch.no_grad():
        for i in range(0, len(Xc), 8192):
            c = torch.tensor(Xc[i:i + 8192], dtype=torch.long, device=device)
            n = torch.tensor(Xn[i:i + 8192], dtype=torch.float32, device=device)
            chunks.append(model(c, n).cpu().numpy())
    return np.concatenate(chunks, axis=0)


def predict_frame(frame):
    return np.maximum(np.expm1(predict_log_frame(frame)), 0.0)


def eval_block(y, p):
    lo, med, hi = p[:, 0], p[:, 1], p[:, 2]
    inside = (y >= lo) & (y <= hi)
    width = hi - lo
    ae = np.abs(y - med)
    bias = med - y
    q = np.asarray(QUANTILES)[None, :]
    e = y[:, None] - p
    return {
        "n": int(len(y)),
        "coverage": float(inside.mean()),
        "mae": float(ae.mean()),
        "median_ae": float(np.median(ae)),
        "bias": float(bias.mean()),
        "median_width": float(np.median(width)),
        "mean_width": float(np.mean(width)),
        "mean_pinball": float(np.mean(np.maximum(q * e, (q - 1) * e))),
        "crossings": int(((lo > med) | (med > hi) | (lo > hi)).sum()),
    }


def interval_metrics(y, p):
    m = eval_block(y, p)
    return {
        "n": m["n"],
        "coverage_90": m["coverage"],
        "mae": m["mae"],
        "median_ae": m["median_ae"],
        "median_bias": m["bias"],
        "median_width": m["median_width"],
        "mean_width": m["mean_width"],
        "crossings": m["crossings"],
    }


def spearman(a, b):
    return float(pd.Series(a).rank().corr(pd.Series(b).rank()))


def topk_overlap(a, b, k=5):
    return len(set(a[:k]) & set(b[:k]))


class FlatModel(torch.nn.Module):
    # Run the trained network from a flattened numeric + embedding input.

    def __init__(self, model, emb_dims, n_num):
        super().__init__()
        self.model = model
        self.emb_dims = emb_dims
        self.n_num = n_num

    def forward(self, x):
        num = x[:, :self.n_num]
        start = self.n_num
        embs = []
        for d in self.emb_dims:
            embs.append(x[:, start:start + d])
            start += d
        h = torch.cat(embs + [num], dim=1)
        h = self.model.encoder.input_layer(h)
        h = self.model.encoder.residual_blocks(h)
        return self.model.head(h)


def flatten_frame(frame):
    Xc, Xn = encode_frame(frame)
    blocks = [Xn]
    for j, (_, dim) in enumerate(prep["embedding_sizes"]):
        idx = torch.tensor(Xc[:, j], dtype=torch.long, device=device)
        blocks.append(model.encoder.embeddings[j].weight[idx].detach().cpu().numpy())
    return np.concatenate(blocks, axis=1).astype(np.float32)


def grouped_shap(sv):
    n_num = len(NUMERIC_FEATURES)
    out = [sv[:, :n_num, :]]
    start = n_num
    for _, dim in prep["embedding_sizes"]:
        out.append(sv[:, start:start + dim, :].sum(axis=1, keepdims=True))
        start += dim
    assert start == sv.shape[1], (start, sv.shape)
    return np.concatenate(out, axis=1)


def encode_compat(frame, float_cols):
    # Encode a normalized frame after restoring the '.0' float-string format.
    cat = pd.DataFrame(index=frame.index)
    for f in CATEGORICAL_FEATURES:
        if f in float_cols:
            cat[f] = pd.to_numeric(frame[f], errors="coerce").astype(float).astype(str)
        else:
            cat[f] = frame[f].astype(str)
    Xc = prep["categorical_encoder"].transform(
        prep["categorical_imputer"].transform(cat)
    ).astype(np.int64) + 1
    Xn = prep["numeric_scaler"].transform(
        prep["numeric_imputer"].transform(frame[NUMERIC_FEATURES])
    ).astype(np.float32)
    return Xc, Xn


def predict_compat(frame, float_cols):
    Xc, Xn = encode_compat(frame, float_cols)
    chunks = []
    with torch.no_grad():
        for i in range(0, len(Xc), 8192):
            c = torch.tensor(Xc[i:i + 8192], dtype=torch.long, device=device)
            n = torch.tensor(Xn[i:i + 8192], dtype=torch.float32, device=device)
            chunks.append(model(c, n).cpu().numpy())
    return np.maximum(np.expm1(np.concatenate(chunks, axis=0)), 0.0)


In [ ]:
# ---------------------------------------------------------------------------
# GradientSHAP on a fixed sample of held-out rows.
# ---------------------------------------------------------------------------
import time

t_shap = time.time()
rng = np.random.default_rng(20261027)
test_idx = np.sort(rng.choice(len(test_df), size=min(SHAP_N_TEST, len(test_df)), replace=False))
bg_idx = np.sort(rng.choice(len(train_df), size=min(SHAP_N_BG, len(train_df)), replace=False))

shap_test = test_df.iloc[test_idx].reset_index(drop=True)
shap_bg = train_df.iloc[bg_idx].reset_index(drop=True)
y_shap = shap_test[TARGET].to_numpy(np.float64)

p_log = predict_log_frame(shap_test)
p_wage = np.maximum(np.expm1(p_log), 0.0)
base_values = predict_log_frame(shap_bg).mean(axis=0)
shap_base = eval_block(y_shap, p_wage)

F_bg = torch.tensor(flatten_frame(shap_bg), dtype=torch.float32, device=device)
F_te = torch.tensor(flatten_frame(shap_test), dtype=torch.float32, device=device)
flat = FlatModel(
    model, [d for _, d in prep["embedding_sizes"]], len(NUMERIC_FEATURES)
).to(device).eval()
explainer = shap.GradientExplainer(flat, F_bg, batch_size=50)
sv = explainer.shap_values(F_te, nsamples=SHAP_NSAMPLES, rseed=0)
if isinstance(sv, list):
    sv = np.stack(sv, axis=-1)
print("GradientSHAP shape:", sv.shape, "in", f"{time.time() - t_shap:.1f}s")

gs = grouped_shap(sv)  # (rows, 34 features, 3 quantiles)

completeness = {}
for k, name in HEADS.items():
    err = gs[:, :, k].sum(axis=1) - (p_log[:, k] - base_values[k])
    completeness[name] = {
        "base_value": float(base_values[k]),
        "mean_error": float(err.mean()),
        "mae_error": float(np.abs(err).mean()),
        "max_abs_error": float(np.abs(err).max()),
        "corr_sum_vs_delta": float(np.corrcoef(
            gs[:, :, k].sum(axis=1), p_log[:, k] - base_values[k]
        )[0, 1]),
        "nsamples": int(SHAP_NSAMPLES),
        "n_background": int(SHAP_N_BG),
    }
print("Completeness:")
print(pd.DataFrame(completeness).T.to_string())

# Feature importance in log1p space.
importance_rows = []
for k, name in HEADS.items():
    z = p_log[:, k]
    effect = np.expm1(z)[:, None] - np.expm1(z[:, None] - gs[:, :, k])
    mean_abs = np.abs(gs[:, :, k]).mean(axis=0)
    mean_signed = gs[:, :, k].mean(axis=0)
    mean_effect = np.abs(effect).mean(axis=0)
    order = np.argsort(-mean_abs)
    total = mean_abs.sum()
    for rank, j in enumerate(order, start=1):
        importance_rows.append({
            "head": name,
            "quantile": QUANTILES[k],
            "feature": FEATURES[j],
            "kind": "categorical" if j >= len(NUMERIC_FEATURES) else "numeric",
            "rank": rank,
            "mean_abs_shap_log1p": float(mean_abs[j]),
            "mean_signed_shap_log1p": float(mean_signed[j]),
            "share": float(mean_abs[j] / total),
            "mean_abs_wage_effect_inr": float(mean_effect[j]),
        })
shap_importance = pd.DataFrame(importance_rows)
shap_importance.to_csv(OUT_DIR / "shap_importance.csv", index=False)

np.savez_compressed(
    OUT_DIR / "shap_values.npz",
    shap_grouped=gs.astype(np.float32),
    features=np.array(FEATURES),
    base_values=base_values.astype(np.float32),
    pred_log=p_log.astype(np.float32),
    pred_wage=p_wage.astype(np.float32),
    y=y_shap.astype(np.float32),
    test_row_index=test_idx,
)

print("\nTop median-wage features:")
print(
    shap_importance[(shap_importance["head"] == "q50") & (shap_importance["rank"] <= 10)]
    [["feature", "share", "mean_abs_shap_log1p", "mean_abs_wage_effect_inr"]]
    .to_string(index=False, float_format=lambda x: f"{x:10.4f}")
)


In [ ]:
# ---------------------------------------------------------------------------
# Permutation and ablation checks on the full held-out test split.
# ---------------------------------------------------------------------------
t_checks = time.time()
full_test = test_df.reset_index(drop=True)
y_full = full_test[TARGET].to_numpy(np.float64)
pred_base = predict_frame(full_test)
m_base = eval_block(y_full, pred_base)

train_median = {f: float(train_df[f].median()) for f in NUMERIC_FEATURES}
train_mode = {f: str(train_df[f].astype(str).mode().iloc[0]) for f in CATEGORICAL_FEATURES}

abl_rows = []
perm_rows = []
for j, feature in enumerate(FEATURES):
    # Ablation: replace one column with its training median/mode.
    alt = full_test.copy()
    if feature in NUMERIC_FEATURES:
        alt[feature] = train_median[feature]
    else:
        alt[feature] = train_mode[feature]
    p_alt = predict_frame(alt)
    m_alt = eval_block(y_full, p_alt)
    delta_median = p_alt[:, 1] - pred_base[:, 1]
    abl_rows.append({
        "feature": feature,
        "kind": "numeric" if feature in NUMERIC_FEATURES else "categorical",
        "delta_mae": m_alt["mae"] - m_base["mae"],
        "delta_coverage": m_alt["coverage"] - m_base["coverage"],
        "delta_mean_pinball": m_alt["mean_pinball"] - m_base["mean_pinball"],
        "mean_abs_delta_median": float(np.abs(delta_median).mean()),
        "crossings": m_alt["crossings"],
    })

    # Permutation: shuffle one raw column.
    alt = full_test.copy()
    rng_p = np.random.default_rng(10_000 + 97 * j)
    alt[feature] = alt[feature].to_numpy()[rng_p.permutation(len(alt))]
    p_alt = predict_frame(alt)
    m_alt = eval_block(y_full, p_alt)
    perm_rows.append({
        "feature": feature,
        "kind": "numeric" if feature in NUMERIC_FEATURES else "categorical",
        "delta_mae": m_alt["mae"] - m_base["mae"],
        "delta_coverage": m_alt["coverage"] - m_base["coverage"],
        "delta_mean_pinball": m_alt["mean_pinball"] - m_base["mean_pinball"],
    })

abl_df = pd.DataFrame(abl_rows)
perm_df = pd.DataFrame(perm_rows)
abl_df.to_csv(OUT_DIR / "shap_ablation.csv", index=False)
perm_df.to_csv(OUT_DIR / "shap_permutation.csv", index=False)

shap_vec = np.abs(gs[:, :, 1]).mean(axis=0)
perm_vec = perm_df["delta_mae"].to_numpy()
abl_vec = abl_df["delta_mae"].to_numpy()
cross = {
    "spearman_shap_vs_permutation": spearman(shap_vec, perm_vec),
    "spearman_shap_vs_ablation": spearman(shap_vec, abl_vec),
    "spearman_permutation_vs_ablation": spearman(perm_vec, abl_vec),
    "top5_shap": [FEATURES[i] for i in np.argsort(-shap_vec)[:5]],
    "top5_permutation": [FEATURES[i] for i in np.argsort(-perm_vec)[:5]],
    "top5_ablation": [FEATURES[i] for i in np.argsort(-abl_vec)[:5]],
    "top5_overlap_shap_permutation": topk_overlap(
        [FEATURES[i] for i in np.argsort(-shap_vec)],
        [FEATURES[i] for i in np.argsort(-perm_vec)], 5),
    "top5_overlap_shap_ablation": topk_overlap(
        [FEATURES[i] for i in np.argsort(-shap_vec)],
        [FEATURES[i] for i in np.argsort(-abl_vec)], 5),
    "top5_overlap_permutation_ablation": topk_overlap(
        [FEATURES[i] for i in np.argsort(-perm_vec)],
        [FEATURES[i] for i in np.argsort(-abl_vec)], 5),
    "top10_overlap_permutation_ablation": topk_overlap(
        [FEATURES[i] for i in np.argsort(-perm_vec)],
        [FEATURES[i] for i in np.argsort(-abl_vec)], 10),
}

shap_summary = {
    "meta": {
        "method": "GradientSHAP on 7 numeric + 27 embedding blocks",
        "n_test_explained": int(len(shap_test)),
        "n_background": int(SHAP_N_BG),
        "nsamples": int(SHAP_NSAMPLES),
        "seed": 20261027,
        "note": "GradientSHAP is approximate; permutation and ablation are independent checks.",
    },
    "sample_baseline": shap_base,
    "full_test_baseline": m_base,
    "completeness": completeness,
    "top_features": {
        name: {
            "top10": shap_importance[shap_importance["head"] == name]
                     .sort_values("rank")["feature"].head(10).tolist(),
            "top5_wage_effect_inr": shap_importance[shap_importance["head"] == name]
                                     .sort_values("rank")
                                     [["feature", "mean_abs_wage_effect_inr"]]
                                     .head(5).to_dict("records"),
        }
        for name in HEADS.values()
    },
    "cross_check": cross,
}
dump_json(shap_summary, OUT_DIR / "shap_summary.json")
print("\nCross-checks:")
print(json.dumps(to_py(cross), indent=2))
print(f"SHAP/permutation/ablation checks completed in {time.time() - t_checks:.1f}s")


## 12. Robustness, uncertainty and schema drift

This section measures how the trained model behaves when the inputs change
slightly, when they are outside the training range, and when categorical codes
lose their original string formatting.

The schema-drift experiment is self-contained: it converts the float-string
categorical columns to integer dtype in memory, scores the normalized frame,
then restores the float-string representation and scores again.


In [ ]:
# ---------------------------------------------------------------------------
# Baseline, tail slices, bootstrap and perturbations.
# ---------------------------------------------------------------------------
y_test = test_df[TARGET].to_numpy(np.float64)
p_test = np.column_stack([test_lower, test_median, test_upper])
baseline_block = eval_block(y_test, p_test)

tails = {}
for label, threshold in [("gt_50k", 50_000), ("gt_100k", 100_000)]:
    m = y_test > threshold
    tails[label] = interval_metrics(y_test[m], p_test[m])
    print(f"{label}: n={m.sum():,} coverage={tails[label]['coverage_90']:.4%} "
          f"MAE=Rs {tails[label]['mae']:,.0f} bias=Rs {tails[label]['median_bias']:,.0f} "
          f"median_width=Rs {tails[label]['median_width']:,.0f}")


def bootstrap_ci(y, p, n_boot=500, seed=20261027):
    rng_b = np.random.default_rng(seed)
    n = len(y)
    lo, med, hi = p[:, 0], p[:, 1], p[:, 2]
    observed = {
        "coverage": float(np.mean((y >= lo) & (y <= hi))),
        "mae": float(np.abs(y - med).mean()),
        "bias": float((med - y).mean()),
        "median_width": float(np.median(hi - lo)),
    }
    cov, mae, bias, width = [], [], [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, n, n)
        lo_i, med_i, hi_i = p[i, 0], p[i, 1], p[i, 2]
        yi = y[i]
        cov.append(np.mean((yi >= lo_i) & (yi <= hi_i)))
        mae.append(np.abs(yi - med_i).mean())
        bias.append((med_i - yi).mean())
        width.append(np.median(hi_i - lo_i))
    out = {}
    for name, arr in (("coverage", cov), ("mae", mae), ("bias", bias), ("median_width", width)):
        a = np.asarray(arr)
        out[name] = {
            "point": observed[name],
            "bootstrap_mean": float(a.mean()),
            "ci95_low": float(np.quantile(a, 0.025)),
            "ci95_high": float(np.quantile(a, 0.975)),
        }
    return out


bootstrap = bootstrap_ci(y_test, p_test, n_boot=N_BOOT)
print("\nBootstrap 95% CI:")
print(json.dumps(to_py(bootstrap), indent=2))

# Fixed perturbation sample.
rng = np.random.default_rng(20261027)
idx = np.sort(rng.choice(len(test_df), size=min(5000, len(test_df)), replace=False))
sample = test_df.iloc[idx].reset_index(drop=True)
ys = sample[TARGET].to_numpy(np.float64)
ps = predict_frame(sample)
base = eval_block(ys, ps)
base_med = ps[:, 1]


def shift_numeric(frame, feature, delta, lo, hi):
    out = frame.copy()
    out[feature] = np.clip(pd.to_numeric(out[feature], errors="coerce") + delta, lo, hi)
    if feature == "potential_experience":
        out["potential_experience_sq"] = out["potential_experience"] ** 2
    return out


def set_numeric(frame, feature, value):
    out = frame.copy()
    out[feature] = value
    if feature == "potential_experience":
        out["potential_experience_sq"] = value ** 2
    return out


rows = []


def add(name, kind, feature, frame):
    pn = predict_frame(frame)
    m = eval_block(ys, pn)
    delta = pn[:, 1] - base_med
    rows.append({
        "perturbation": name,
        "kind": kind,
        "feature": feature,
        "n": int(len(ys)),
        "mae_base": base["mae"],
        "mae_new": m["mae"],
        "delta_mae": m["mae"] - base["mae"],
        "coverage_base": base["coverage"],
        "coverage_new": m["coverage"],
        "delta_coverage": m["coverage"] - base["coverage"],
        "delta_pinball": m["mean_pinball"] - base["mean_pinball"],
        "crossings_new": m["crossings"],
        "mean_abs_delta_median": float(np.abs(delta).mean()),
        "p95_abs_delta_median": float(np.quantile(np.abs(delta), 0.95)),
    })


for name, feature, delta, lo, hi in (
    ("age_plus1", "age", 1, 15, 80),
    ("age_plus5", "age", 5, 15, 80),
    ("education_plus1", "years_formal_education", 1, 0, 20),
    ("experience_plus5", "potential_experience", 5, 0, 70),
    ("hours_plus5", "tothrs_wrk", 5, 0, 168),
    ("days_plus1", "days_worked", 1, 0, 7),
):
    add(name, "numeric_shift", feature, shift_numeric(sample, feature, delta, lo, hi))

jitter = sample.copy()
rng_j = np.random.default_rng(4242)
n = len(jitter)
jitter["age"] = np.clip(jitter["age"] + rng_j.integers(-1, 2, n), 15, 80)
jitter["years_formal_education"] = np.clip(
    jitter["years_formal_education"] + rng_j.integers(-1, 2, n), 0, 20)
jitter["potential_experience"] = np.clip(
    jitter["potential_experience"] + rng_j.integers(-1, 2, n), 0, 70)
jitter["potential_experience_sq"] = jitter["potential_experience"] ** 2
jitter["tothrs_wrk"] = np.clip(jitter["tothrs_wrk"] + rng_j.integers(-2, 3, n), 0, 168)
jitter["days_worked"] = np.clip(jitter["days_worked"] + rng_j.integers(-1, 2, n), 0, 7)
add("realistic_jitter", "numeric_jitter", "all_numeric", jitter)

for name, feature, value in (
    ("ood_age_100", "age", 100),
    ("ood_days_31", "days_worked", 31),
    ("ood_all_extreme", "all_numeric", None),
):
    if name == "ood_all_extreme":
        alt = sample.copy()
        for f, v in (("age", 100), ("years_formal_education", 25),
                     ("potential_experience", 80), ("tothrs_wrk", 168),
                     ("days_worked", 31), ("avg_daily_hours", 24)):
            alt[f] = v
        alt["potential_experience_sq"] = 6400
        add(name, "out_of_range", feature, alt)
    else:
        add(name, "out_of_range", feature, set_numeric(sample, feature, value))

flip = sample.copy()
s = flip["sex"].to_numpy()
flip["sex"] = np.where(s == 1, 2, np.where(s == 2, 1, s))
add("sex_flip", "categorical_flip", "sex", flip)

donor = rng.integers(0, len(sample), len(sample))
geo = sample.copy()
for feature in ("st", "dc", "state_district"):
    geo[feature] = sample[feature].to_numpy()[donor]
add("geo_swap_consistent", "categorical_swap", "st+dc+state_district", geo)

for name, feature in [
    ("missing_ssec_pas", "ssec_pas"),
    ("oov_ssec_pas", "ssec_pas"),
    ("oov_ocu_pas", "ocu_pas"),
]:
    alt = sample.copy()
    if name.startswith("missing"):
        alt[feature] = np.nan
        add(name, "missing_category", feature, alt)
    else:
        alt[feature] = "__UNSEEN__"
        add(name, "unknown_category", feature, alt)

all_unknown = sample.copy()
for feature in CATEGORICAL_FEATURES:
    all_unknown[feature] = "__UNSEEN__"
add("all_cat_unknown", "unknown_category", "all_categorical", all_unknown)

all_missing = sample.copy()
for feature in CATEGORICAL_FEATURES:
    all_missing[feature] = np.nan
add("all_cat_missing", "missing_category", "all_categorical", all_missing)

perturbations = pd.DataFrame(rows)
perturbations.to_csv(OUT_DIR / "wage_perturbations.csv", index=False)
print("\nSelected perturbations:")
print(
    perturbations[perturbations["perturbation"].isin([
        "age_plus1", "age_plus5", "education_plus1", "experience_plus5",
        "hours_plus5", "days_plus1", "realistic_jitter", "ood_age_100",
        "ood_days_31", "all_cat_missing", "all_cat_unknown",
    ])][["perturbation", "kind", "mean_abs_delta_median", "delta_mae",
         "delta_coverage", "crossings_new"]]
    .to_string(index=False, float_format=lambda x: f"{x:10.3f}")
)

# MC-dropout stability (not a calibrated interval).
mc_idx = np.sort(np.random.default_rng(777).choice(
    len(test_df), size=min(1000, len(test_df)), replace=False))
mc = test_df.iloc[mc_idx].reset_index(drop=True)
model.train()
passes = []
with torch.no_grad():
    for _ in range(MC_PASSES):
        passes.append(predict_log_frame(mc))
model.eval()
S = np.stack(passes, axis=0)
W = np.expm1(S)
mc_dropout = {
    "n": int(len(mc)),
    "passes": int(MC_PASSES),
    "median_head_log_std_mean": float(S.std(axis=0)[:, 1].mean()),
    "median_head_wage_std_mean": float(W.std(axis=0)[:, 1].mean()),
    "median_head_wage_std_p95": float(np.quantile(W.std(axis=0)[:, 1], 0.95)),
    "note": "Dropout active; not a calibrated uncertainty interval.",
}
print("\nMC dropout:", json.dumps(to_py(mc_dropout), indent=2))


In [ ]:
# ---------------------------------------------------------------------------
# Subgroups, calibration deciles and in-memory schema drift.
# ---------------------------------------------------------------------------
meta = pd.DataFrame({
    "survey_year": test_df["survey_year"].astype(str),
    "sex": test_df["sex"].astype(str),
    "sector": test_df["sec"].astype(str),
    "age_group": pd.cut(
        test_df["age"], bins=[15, 25, 35, 45, 55, 65, 200],
        labels=["15-24", "25-34", "35-44", "45-54", "55-64", "65+"], right=False
    ).astype(str),
    "wage_band": pd.cut(
        y_test, bins=[-1, 10_000, 20_000, 50_000, 100_000, np.inf],
        labels=["0-10k", "10-20k", "20-50k", "50-100k", "100k+"]
    ).astype(str),
    "pred_quintile": pd.qcut(
        p_test[:, 1], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop"
    ).astype(str),
})

sub_rows = []
for dim in ("survey_year", "sex", "sector", "age_group", "wage_band", "pred_quintile"):
    for group in sorted(meta[dim].unique(), key=str):
        m = (meta[dim] == group).to_numpy()
        if m.sum() < 30:
            continue
        met = eval_block(y_test[m], p_test[m])
        sub_rows.append({
            "dimension": dim, "group": group, "n": int(m.sum()),
            "share": float(m.mean()), "coverage": met["coverage"],
            "coverage_gap_vs_overall": met["coverage"] - baseline_block["coverage"],
            "mae": met["mae"], "bias": met["bias"],
            "median_width": met["median_width"],
            "mean_pinball": met["mean_pinball"],
            "crossings": met["crossings"],
        })
subgroups = pd.DataFrame(sub_rows)
subgroups.to_csv(OUT_DIR / "wage_subgroups.csv", index=False)

dec = pd.qcut(pd.Series(test_median), q=10, labels=False, duplicates="drop")
reliability = []
for b in sorted(pd.unique(dec)):
    m = (dec == b).to_numpy()
    reliability.append({
        "decile": int(b) + 1,
        "n": int(m.sum()),
        "mean_pred": float(test_median[m].mean()),
        "mean_actual": float(y_test[m].mean()),
        "coverage": float(inside[m].mean()),
    })
reliability = pd.DataFrame(reliability)
reliability.to_csv(OUT_DIR / "wage_reliability.csv", index=False)
print("Reliability by predicted-median decile:")
print(reliability.to_string(index=False, float_format=lambda x: f"{x:10.2f}"))

# In-memory schema drift: int-normalized categoricals vs float-string shim.
float_cols = []
for j, feature in enumerate(CATEGORICAL_FEATURES):
    cats = [str(c) for c in prep["categorical_encoder"].categories_[j]]
    if np.mean([c.endswith(".0") for c in cats]) > 0.5:
        float_cols.append(feature)

normalized = test_df.copy()
for feature in float_cols:
    normalized[feature] = pd.to_numeric(normalized[feature], errors="coerce").astype("Int64")

p_final_raw = predict_frame(normalized)
p_final_shim = predict_compat(normalized, float_cols)
drift = {
    "rows": {
        "original": int(len(test_df)),
        "normalized_copy": int(len(normalized)),
        "difference": 0,
    },
    "columns": {
        "n_columns": int(test_df.shape[1]),
        "categorical_needing_float_string": float_cols,
    },
    "dev_original": {
        "coverage": baseline_block["coverage"],
        "mae": baseline_block["mae"],
        "bias": baseline_block["bias"],
        "median_width": baseline_block["median_width"],
        "crossings": baseline_block["crossings"],
    },
    "final_raw": interval_metrics(y_test, p_final_raw),
    "final_shim": interval_metrics(y_test, p_final_shim),
    "final_raw_vs_shim": {
        "mean_abs_delta_median": float(np.abs(p_final_shim[:, 1] - p_final_raw[:, 1]).mean()),
        "max_abs_delta_median": float(np.abs(p_final_shim[:, 1] - p_final_raw[:, 1]).max()),
    },
}
dump_json(drift, OUT_DIR / "wage_drift_summary.json")
print("\nSchema drift:")
print(json.dumps(to_py({
    "original_coverage": drift["dev_original"]["coverage"],
    "normalized_raw_coverage": drift["final_raw"]["coverage_90"],
    "normalized_shim_coverage": drift["final_shim"]["coverage_90"],
}), indent=2))


## 13. Presentation figures

The figures below are the slide-ready set:

- `figures/training/` — training history, test metrics, calibration and coverage;
- `figures/presentation/` — metrics, subgroup coverage, calibration, SHAP,
  robustness and schema drift.

They are deliberately simple: one table or one bar/line chart per slide.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PRES = FIG_PRES
PRES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
C_BLUE, C_ORANGE, C_GREEN, C_RED = "#4C78A8", "#F58518", "#54A24B", "#E45756"


def save(fig, name):
    fig.tight_layout()
    fig.savefig(PRES / name, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", PRES / name)


# 1) Headline test metrics table.
rows = [
    ["All", f"{baseline_block['n']:,}",
     f"{baseline_block['coverage']*100:.2f}%",
     f"Rs {baseline_block['mae']:,.0f}",
     f"Rs {baseline_block['bias']:,.0f}",
     f"Rs {baseline_block['median_width']:,.0f}"],
    ["> Rs 50k", f"{tails['gt_50k']['n']:,}",
     f"{tails['gt_50k']['coverage_90']*100:.2f}%",
     f"Rs {tails['gt_50k']['mae']:,.0f}",
     f"Rs {tails['gt_50k']['median_bias']:,.0f}",
     f"Rs {tails['gt_50k']['median_width']:,.0f}"],
    ["> Rs 100k", f"{tails['gt_100k']['n']:,}",
     f"{tails['gt_100k']['coverage_90']*100:.2f}%",
     f"Rs {tails['gt_100k']['mae']:,.0f}",
     f"Rs {tails['gt_100k']['median_bias']:,.0f}",
     f"Rs {tails['gt_100k']['median_width']:,.0f}"],
]
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis("off")
tbl = ax.table(
    cellText=rows,
    colLabels=["Slice", "n", "5-95 coverage", "MAE", "Bias", "Median width"],
    loc="center", cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D9D9D9")
    if r == 0:
        cell.set_facecolor("#EAF2F8")
        cell.set_text_props(weight="bold")
ax.set_title("Final wage model — held-out test metrics", pad=12)
save(fig, "01_test_metrics_table.png")

# 2) Coverage by selected subgroup.
def cov(dim, group):
    r = subgroups[(subgroups["dimension"] == dim) & (subgroups["group"].astype(str) == str(group))]
    return float(r["coverage"].iloc[0]) if len(r) else np.nan

labels = ["Overall", "0-10k", "10-20k", "20-50k", "50-100k", "100k+",
          "Sex 1", "Sex 2", "Age 65+"]
vals = [baseline_block["coverage"],
        cov("wage_band", "0-10k"), cov("wage_band", "10-20k"),
        cov("wage_band", "20-50k"), cov("wage_band", "50-100k"),
        cov("wage_band", "100k+"), cov("sex", 1), cov("sex", 2),
        cov("age_group", "65+")]
fig, ax = plt.subplots(figsize=(8, 5))
colors = [C_BLUE] + [C_ORANGE] * 5 + [C_GREEN] * 3
bars = ax.barh(labels[::-1], [100 * v for v in vals[::-1]], color=colors[::-1], height=0.65)
for b, v in zip(bars, vals[::-1]):
    ax.text(100 * v + 0.3, b.get_y() + b.get_height() / 2, f"{100*v:.1f}%",
            va="center", fontsize=9)
ax.axvline(90, color="black", ls="--", lw=1, label="nominal 90%")
ax.set_xlim(75, 100)
ax.set_xlabel("5-95 coverage (%)")
ax.set_title("Coverage by subgroup and wage band")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "02_coverage_by_group.png")

# 3) Calibration by predicted-median decile.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(reliability["mean_pred"], reliability["mean_actual"], marker="o", lw=2, color=C_BLUE)
lo = min(reliability["mean_pred"].min(), reliability["mean_actual"].min())
hi = max(reliability["mean_pred"].max(), reliability["mean_actual"].max())
axes[0].plot([lo, hi], [lo, hi], "--", color="black", lw=1)
axes[0].set_xlabel("mean predicted median wage (Rs)")
axes[0].set_ylabel("mean actual wage (Rs)")
axes[0].set_title("Calibration by predicted decile")
axes[0].grid(alpha=0.25)
axes[0].spines[["top", "right"]].set_visible(False)
axes[1].bar(reliability["decile"].astype(str), 100 * reliability["coverage"], color=C_GREEN)
axes[1].axhline(90, color="black", ls="--", lw=1)
axes[1].set_xlabel("predicted-median decile")
axes[1].set_ylabel("5-95 coverage (%)")
axes[1].set_title("Coverage by predicted decile")
axes[1].set_ylim(75, 100)
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines[["top", "right"]].set_visible(False)
save(fig, "03_calibration_deciles.png")

# 4) SHAP top-10 for median wages.
d = shap_importance[(shap_importance["head"] == "q50") & (shap_importance["rank"] <= 10)] \
    .sort_values("mean_abs_shap_log1p")
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(d["feature"], d["mean_abs_shap_log1p"], color=C_BLUE, height=0.65)
for b, v, s in zip(bars, d["mean_abs_shap_log1p"], d["share"]):
    ax.text(v + 0.002, b.get_y() + b.get_height() / 2, f"{100*s:.1f}%",
            va="center", fontsize=9)
ax.set_xlabel("mean |SHAP| (log1p wage)")
ax.set_title("SHAP importance — median wage (q50)")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "04_shap_top10_q50.png")

# 5) Top-5 SHAP table for median wages.
d5 = shap_importance[(shap_importance["head"] == "q50") & (shap_importance["rank"] <= 5)] \
    .sort_values("rank")
rows = [[r["feature"], f"{100*r['share']:.1f}%", f"Rs {r['mean_abs_wage_effect_inr']:,.0f}"]
        for _, r in d5.iterrows()]
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.axis("off")
tbl = ax.table(
    cellText=rows,
    colLabels=["Feature", "Share of total |SHAP|", "Approx. rupee effect"],
    loc="center", cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D9D9D9")
    if r == 0:
        cell.set_facecolor("#EAF2F8")
        cell.set_text_props(weight="bold")
ax.set_title("Median-wage SHAP — top 5 features", pad=12)
save(fig, "05_shap_top5_table.png")

# 6) SHAP cross-check table.
cc = shap_summary["cross_check"]
shap_top10 = shap_importance[(shap_importance["head"] == "q50") & (shap_importance["rank"] <= 10)] \
    .sort_values("rank")["feature"].tolist()
perm_top10 = perm_df.sort_values("delta_mae", ascending=False)["feature"].head(10).tolist()
abl_top10 = abl_df.sort_values("delta_mae", ascending=False)["feature"].head(10).tolist()
rows = [
    ["SHAP vs permutation", f"{cc['spearman_shap_vs_permutation']:.3f}",
     f"{cc['top5_overlap_shap_permutation']}/5", f"{len(set(shap_top10) & set(perm_top10))}/10"],
    ["SHAP vs ablation", f"{cc['spearman_shap_vs_ablation']:.3f}",
     f"{cc['top5_overlap_shap_ablation']}/5", f"{len(set(shap_top10) & set(abl_top10))}/10"],
    ["Permutation vs ablation", f"{cc['spearman_permutation_vs_ablation']:.3f}",
     f"{cc['top5_overlap_permutation_ablation']}/5",
     f"{cc['top10_overlap_permutation_ablation']}/10"],
]
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.axis("off")
tbl = ax.table(
    cellText=rows,
    colLabels=["Comparison", "Spearman rho", "Top-5 overlap", "Top-10 overlap"],
    loc="center", cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D9D9D9")
    if r == 0:
        cell.set_facecolor("#EAF2F8")
        cell.set_text_props(weight="bold")
ax.set_title("SHAP cross-checks", pad=12)
save(fig, "06_shap_crosscheck_table.png")

# 7) Realistic robustness perturbations.
real_keys = ["age_plus1", "age_plus5", "education_plus1", "experience_plus5",
             "hours_plus5", "days_plus1", "realistic_jitter"]
pretty = {
    "age_plus1": "Age +1", "age_plus5": "Age +5",
    "education_plus1": "Education +1 yr", "experience_plus5": "Experience +5 yrs",
    "hours_plus5": "Hours +5", "days_plus1": "Days worked +1",
    "realistic_jitter": "Realistic jitter",
}
r = perturbations[perturbations["perturbation"].isin(real_keys)] \
    .set_index("perturbation").loc[real_keys]
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh([pretty[k] for k in real_keys][::-1],
               r["mean_abs_delta_median"].values[::-1],
               color=C_GREEN, height=0.65)
for b, v in zip(bars, r["mean_abs_delta_median"].values[::-1]):
    ax.text(v + 20, b.get_y() + b.get_height() / 2, f"Rs {v:,.0f}", va="center", fontsize=9)
ax.set_xlabel("mean absolute change in predicted median wage (Rs)")
ax.set_title("Robustness — realistic numeric changes")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "07_robustness_realistic.png")

# 8) Failure-case perturbations.
fail_keys = ["missing_ssec_pas", "oov_ssec_pas", "oov_ocu_pas",
             "all_cat_missing", "all_cat_unknown", "ood_age_100", "ood_days_31"]
pretty_fail = {
    "missing_ssec_pas": "ssec_pas missing", "oov_ssec_pas": "ssec_pas unknown",
    "oov_ocu_pas": "ocu_pas unknown", "all_cat_missing": "All categorical missing",
    "all_cat_unknown": "All categorical unknown", "ood_age_100": "Age = 100",
    "ood_days_31": "Days worked = 31",
}
f = perturbations[perturbations["perturbation"].isin(fail_keys)] \
    .set_index("perturbation").loc[fail_keys]
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.barh([pretty_fail[k] for k in fail_keys][::-1],
               f["mean_abs_delta_median"].values[::-1],
               color=C_RED, height=0.65)
for b, v in zip(bars, f["mean_abs_delta_median"].values[::-1]):
    ax.text(v + 150, b.get_y() + b.get_height() / 2, f"Rs {v:,.0f}", va="center", fontsize=9)
ax.set_xlabel("mean absolute change in predicted median wage (Rs)")
ax.set_title("Robustness — out-of-range, missing and unseen inputs")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "08_robustness_failures.png")

# 9) Schema-drift coverage.
labels = ["Original schema", "Normalized copy", "Normalized + shim"]
vals = [100 * drift["dev_original"]["coverage"],
        100 * drift["final_raw"]["coverage_90"],
        100 * drift["final_shim"]["coverage_90"]]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals, color=[C_BLUE, C_RED, C_GREEN], width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)
ax.axhline(90, color="black", ls="--", lw=1, label="nominal 90%")
ax.set_ylim(60, 100)
ax.set_ylabel("5-95 coverage (%)")
ax.set_title("Schema drift — coverage by input representation")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "09_schema_drift.png")


## 14. Save metrics and model card

The training metrics are expanded with the SHAP, robustness, bootstrap and
schema-drift results. The model card records the headline results and the
limitations that should travel with the model.


In [ ]:
final_metrics = {
    "model": "PLFS 2023-2025 weighted 5-50-95 quantile MLP",
    "dataset": str(DATA_PATH.relative_to(ROOT)).replace("\\", "/"),
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "config": metrics["config"],
    "training": metrics["training"],
    "test": metrics["test"],
    "baseline_test": baseline_block,
    "tail_50k": tails["gt_50k"],
    "tail_100k": tails["gt_100k"],
    "bootstrap_ci": bootstrap,
    "mc_dropout": mc_dropout,
    "shap": {
        "sample": shap_base,
        "completeness": completeness,
        "top_features": shap_summary["top_features"],
        "cross_check": cross,
    },
    "robustness_perturbations": int(len(perturbations)),
    "robustness_max_crossings": int(perturbations["crossings_new"].max()),
    "drift": drift,
}
dump_json(final_metrics, OUT_DIR / "metrics.json")

card = f'''# PLFS Wage Model Card

**Status:** reproducible end-to-end notebook result.
**Target:** monthly regular wage of a PLFS regular salaried/wage worker, 2025 prices.
**Outputs:** 5th / 50th / 95th percentile.
**Data:** {len(df):,} rows; train/val/test = {len(train_df):,} / {len(val_df):,} / {len(test_df):,}.

## Test performance

| Slice | n | 5-95 coverage | MAE | Bias | Median width |
|---|---:|---:|---:|---:|---:|
| All | {baseline_block['n']:,} | {baseline_block['coverage']:.2%} | Rs {baseline_block['mae']:,.0f} | Rs {baseline_block['bias']:,.0f} | Rs {baseline_block['median_width']:,.0f} |
| > Rs 50k | {tails['gt_50k']['n']:,} | {tails['gt_50k']['coverage_90']:.2%} | Rs {tails['gt_50k']['mae']:,.0f} | Rs {tails['gt_50k']['median_bias']:,.0f} | Rs {tails['gt_50k']['median_width']:,.0f} |
| > Rs 100k | {tails['gt_100k']['n']:,} | {tails['gt_100k']['coverage_90']:.2%} | Rs {tails['gt_100k']['mae']:,.0f} | Rs {tails['gt_100k']['median_bias']:,.0f} | Rs {tails['gt_100k']['median_width']:,.0f} |

Quantile crossings: {baseline_block['crossings']}. Mean pinball: Rs {baseline_block['mean_pinball']:,.0f}.

## SHAP

Top median-wage features: {', '.join(shap_summary['top_features']['q50']['top10'][:5])}.
SHAP vs permutation Spearman: {cross['spearman_shap_vs_permutation']:.3f};
SHAP vs ablation Spearman: {cross['spearman_shap_vs_ablation']:.3f}.

## Limitations

- High-wage tail undercoverage: > Rs 100k coverage {tails['gt_100k']['coverage_90']:.2%}, bias Rs {tails['gt_100k']['median_bias']:,.0f}.
- Categorical codes must keep their original string formatting; normalizing them in memory drops coverage from {drift['dev_original']['coverage']:.2%} to {drift['final_raw']['coverage_90']:.2%}, while the float-string shim restores it to {drift['final_shim']['coverage_90']:.2%}.
- No input-range validation: `days_worked=31` and `age=100` are high-risk.
- Unknown categories map to an untrained embedding.
- GradientSHAP is approximate, not exact Shapley value.
- Weighted quantile intervals are not conformal; 90% coverage is empirical.
- SHAP and robustness are model-behaviour diagnostics, not causal estimates.

## Artifacts

`wage_quantile_mlp.pt`, `wage_preprocessing.pkl`, `model_info.json`, `metrics.json`,
`test_predictions.csv.gz`, `shap_importance.csv`, `shap_permutation.csv`,
`shap_ablation.csv`, `wage_perturbations.csv`, `wage_subgroups.csv`,
`wage_reliability.csv`, `wage_drift_summary.json`.
'''
(OUT_DIR / "model_card.md").write_text(card, encoding="utf-8")
print(card)

print("\nFinal artifacts:")
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:45s} {p.stat().st_size:>10,}")
print("\nSlide figures:")
for p in sorted(FIG_TRAIN.glob("*.png")) + sorted(FIG_PRES.glob("*.png")):
    print(f"  {p.relative_to(ROOT)}")


## 15. How to use the outputs

- `models/wage/wage_quantile_mlp.pt` — trained weights for deployment or further evaluation.
- `models/wage/wage_preprocessing.pkl` — fitted imputers, scaler and ordinal encoder.
- `models/wage/test_predictions.csv.gz` — row-level test predictions.
- `models/wage/metrics.json` — full metrics, SHAP cross-checks, robustness and drift.
- `figures/training/` and `figures/presentation/` — PNGs for slides.

The model is intentionally evaluated as a **weighted quantile interval**, not a
conformal predictor. The 5-95 interval has empirical 90% coverage on the test
split; it is not a guarantee for new populations.
